In [92]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import sklearn as sk 

In [93]:

import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Load the dataset
data = pd.read_csv('hand_angles_datasets.csv')

In [94]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31926 entries, 0 to 31925
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   label                    31926 non-null  object 
 1   both_hands               31926 non-null  int64  
 2   thumb_Left               31926 non-null  float64
 3   index_finger_Left        31926 non-null  float64
 4   middle_finger_Left       31926 non-null  float64
 5   ring_finger_Left         31926 non-null  float64
 6   pinky_Left               31926 non-null  float64
 7   palm_angle_Left_left     31926 non-null  float64
 8   palm_angle_Left_right    31926 non-null  float64
 9   hand_Left_ground_angle   31926 non-null  float64
 10  thumb_Right              20540 non-null  float64
 11  index_finger_Right       20540 non-null  float64
 12  middle_finger_Right      20540 non-null  float64
 13  ring_finger_Right        20540 non-null  float64
 14  pinky_Right           

In [95]:
data.describe()

,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
count,31926.000000,31926.000000,31926.000000,31926.000000,31926.000000,31926.000000,31926.000000,31926.000000,31926.000000,20540.000000,20540.000000,20540.000000,20540.000000,20540.000000,20540.000000,20540.000000,20540.000000
mean,0.642141,161.119467,142.782505,95.987261,80.919465,86.096034,123.382234,97.892115,97.723929,158.921079,143.591855,98.966288,77.394371,91.207821,119.993806,109.663086,72.038210
std,0.479378,18.255605,46.270904,59.333207,59.296842,58.882588,28.121245,39.197875,52.048830,14.028742,53.459499,62.015123,54.980801,53.528091,19.668325,38.855860,44.263625
min,0.000000,0.169815,0.038675,0.077729,0.006752,0.004763,1.759072,0.018918,0.003967,3.051912,0.002837,0.011064,0.000543,0.001149,14.922736,1.102502,0.020284
25%,0.000000,154.909142,116.705122,41.452154,30.313899,36.111476,102.034198,73.734086,56.816134,151.435730,141.252381,55.803766,43.329852,54.554766,103.722691,75.150869,39.723237
50%,1.000000,164.035143,170.157503,81.550656,62.842380,75.991861,117.759712,83.775232,104.835236,159.412490,171.372483,68.127911,56.631255,74.012659,116.801114,97.309502,74.999347
75%,1.000000,171.921401,175.733766,172.193546,127.373674,142.278357,145.257109,127.361825,140.154974,169.649495,175.470057,175.407882,107.094089,132.201191,137.586656,141.905859,95.263768
max,1.000000,179.999547,179.999708,179.999728,179.999730,179.999434,179.997447,179.996875,179.985985,179.996462,179.995613,179.999563,179.999456,179.993423,179.969215,179.997683,179.994367


In [96]:
data.isnull().sum()

label                          0
both_hands                     0
thumb_Left                     0
index_finger_Left              0
middle_finger_Left             0
ring_finger_Left               0
pinky_Left                     0
palm_angle_Left_left           0
palm_angle_Left_right          0
hand_Left_ground_angle         0
thumb_Right                11386
index_finger_Right         11386
middle_finger_Right        11386
ring_finger_Right          11386
pinky_Right                11386
palm_angle_Right_left      11386
palm_angle_Right_right     11386
hand_Right_ground_angle    11386
dtype: int64

In [97]:
def impute_missing_values(data):
    for col in data.columns:
        if data[col].dtype in ['int64', 'float64']:
            data[col].fillna(data[col].mean(), inplace=True)
    return data
impute_missing_values(data)

,label,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
0,A,1,165.866894,9.472022,7.338085,8.883946,11.885869,130.748954,9.044749,118.313201,158.691070,26.324367,23.344084,16.753552,15.052806,138.827512,80.243915,54.198266
1,A,1,170.276031,27.749708,19.730523,13.508492,8.662417,137.910787,31.028064,120.761914,160.601171,23.365473,17.301898,15.469334,11.218169,143.362799,94.906408,50.830493
2,A,1,170.240058,27.665807,19.216161,12.458123,7.245331,138.978871,25.260721,124.087366,166.259334,22.784136,16.674488,14.218492,8.793953,141.223270,117.228873,55.208722
3,A,1,166.149868,24.753762,17.219777,9.735988,5.652476,142.815529,15.835364,131.335872,158.704965,21.870556,8.205465,3.303481,2.859875,161.010664,71.219133,41.282047
4,A,1,161.666006,18.942529,8.619889,0.276340,4.349829,165.708388,60.330268,150.884244,164.216495,14.128659,7.689510,0.507705,3.929584,135.711526,45.242899,50.204542
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31921,Z,1,178.223334,170.474969,175.532100,177.994736,171.909614,101.383514,138.831761,109.866122,162.109073,176.274971,176.173612,173.179314,170.227311,107.851029,174.780808,81.241860
31922,Z,1,179.070333,170.766332,177.477504,177.044186,171.361848,101.112184,145.046158,108.712098,161.672697,176.300226,176.299349,172.897940,170.586520,107.187214,176.535478,82.135254
31923,Z,1,178.447356,170.297645,176.330413,178.249619,171.950810,100.660647,146.378833,109.715095,165.186535,176.070507,176.138492,172.759832,170.120754,109.447518,167.774662,81.010436
31924,Z,1,177.970894,170.598253,176.457503,177.126304,171.228011,101.398491,142.622573,110.309263,163.031021,176.406600,176.877823,173.319362,170.379484,106.792592,178.160676,82.794015


In [98]:
data.tail()

,label,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
31921,Z,1,178.223334,170.474969,175.532100,177.994736,171.909614,101.383514,138.831761,109.866122,162.109073,176.274971,176.173612,173.179314,170.227311,107.851029,174.780808,81.241860
31922,Z,1,179.070333,170.766332,177.477504,177.044186,171.361848,101.112184,145.046158,108.712098,161.672697,176.300226,176.299349,172.897940,170.586520,107.187214,176.535478,82.135254
31923,Z,1,178.447356,170.297645,176.330413,178.249619,171.950810,100.660647,146.378833,109.715095,165.186535,176.070507,176.138492,172.759832,170.120754,109.447518,167.774662,81.010436
31924,Z,1,177.970894,170.598253,176.457503,177.126304,171.228011,101.398491,142.622573,110.309263,163.031021,176.406600,176.877823,173.319362,170.379484,106.792592,178.160676,82.794015
31925,Z,1,179.357116,170.904881,176.210377,177.409241,171.986656,101.054316,141.937561,110.330033,164.813063,175.105790,175.126220,171.593307,168.815382,107.076756,175.551137,82.379906


In [99]:
data.isnull().sum()

label                      0
both_hands                 0
thumb_Left                 0
index_finger_Left          0
middle_finger_Left         0
ring_finger_Left           0
pinky_Left                 0
palm_angle_Left_left       0
palm_angle_Left_right      0
hand_Left_ground_angle     0
thumb_Right                0
index_finger_Right         0
middle_finger_Right        0
ring_finger_Right          0
pinky_Right                0
palm_angle_Right_left      0
palm_angle_Right_right     0
hand_Right_ground_angle    0
dtype: int64

In [100]:
def find_outliers(data, column):
    for col in data.columns:
        if data[col].dtype in ['int64', 'float64']:
            Q1 = data[col].quantile(0.25)
            Q3 = data[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
            print(f'Outliers in {col}:')
            print(outliers)
    return outliers
find_outliers(data, 'x1')

Outliers in both_hands:
Empty DataFrame
Columns: [label, both_hands, thumb_Left, index_finger_Left, middle_finger_Left, ring_finger_Left, pinky_Left, palm_angle_Left_left, palm_angle_Left_right, hand_Left_ground_angle, thumb_Right, index_finger_Right, middle_finger_Right, ring_finger_Right, pinky_Right, palm_angle_Right_left, palm_angle_Right_right, hand_Right_ground_angle]
Index: []
Outliers in thumb_Left:
      label  both_hands  thumb_Left  index_finger_Left  middle_finger_Left  \
1321      B           1  128.326619          92.283999           90.950215   
1350      B           1  127.811673          91.998201           90.062429   
1352      B           1  128.900644          93.726942           88.642543   
1353      B           1  128.538414          93.550585           87.102506   
1355      B           1  127.193859          91.562878           88.835573   
...     ...         ...         ...                ...                 ...   
29484     X           0  117.251027        

,label,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
238,A,1,158.811017,13.958126,11.690246,5.087919,4.779535,177.135444,58.194795,171.242408,153.504102,17.965848,8.616601,1.444311,4.859075,168.902656,67.799265,30.449076
244,A,1,159.923962,12.990367,9.163288,5.013148,3.990141,177.320889,58.095760,173.212556,152.872465,15.907216,8.539273,2.220407,5.707739,168.513041,68.197203,32.439128
248,A,1,160.814051,10.849036,8.967961,5.864040,1.028235,178.426250,54.347962,170.869894,150.983962,11.859460,6.297437,2.668116,9.668405,174.264281,69.310741,31.559393
249,A,1,160.489127,13.120204,10.869891,4.963121,5.469715,173.215383,55.425150,175.994266,150.553686,16.672433,8.844289,1.885017,6.819601,170.103585,67.691206,32.215595
250,A,1,157.711442,12.109856,10.168940,5.498638,5.643317,177.183233,57.777023,175.633006,152.042707,13.830697,8.299571,2.135040,6.493621,170.266286,66.728971,31.449940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29164,X,1,167.827028,175.883077,33.906383,22.665143,27.369632,171.134796,119.505007,154.944206,128.389086,168.516408,34.085256,15.477968,9.591733,157.452504,134.916021,30.110486
29165,X,1,166.722584,176.380456,34.304770,21.684862,25.626226,172.449308,124.740649,152.579177,134.002404,168.231109,34.219078,16.770235,10.717209,157.552471,136.892625,31.055876
29166,X,1,166.250680,176.312940,33.474800,23.663976,27.928698,173.968099,125.340741,152.314747,132.584017,168.046395,33.429984,16.067626,10.953741,153.939847,141.482527,32.331945
29167,X,1,165.332367,173.883572,34.603737,23.930625,26.710433,171.954194,123.795563,153.109996,125.703140,166.817329,34.173112,17.226504,12.914524,152.471021,141.471557,32.537708


In [101]:
def normalize_data(data):
    for col in data.columns:
        if data[col].dtype in ['int64', 'float64']:
            data[col] = (data[col] - data[col].min()) / (data[col].max() - data[col].min()) ##min-max normalization
    return data
normalize_data(data)

,label,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
0,A,1.0,0.921411,0.052419,0.040353,0.049320,0.066008,0.723693,0.050150,0.657339,0.879593,0.146237,0.129636,0.093073,0.083624,0.750727,0.442390,0.301032
1,A,1.0,0.945929,0.153984,0.109230,0.075013,0.048100,0.763874,0.172294,0.670945,0.890388,0.129798,0.096066,0.085938,0.062319,0.778205,0.524351,0.282320
2,A,1.0,0.945729,0.153517,0.106371,0.069177,0.040227,0.769867,0.140249,0.689421,0.922365,0.126568,0.092580,0.078989,0.048851,0.765242,0.649131,0.306647
3,A,1.0,0.922984,0.137336,0.095275,0.054053,0.031377,0.791392,0.087880,0.729695,0.879671,0.121492,0.045527,0.018350,0.015882,0.885132,0.391943,0.229265
4,A,1.0,0.898051,0.105044,0.047477,0.001498,0.024140,0.919832,0.335104,0.838308,0.910820,0.078480,0.042661,0.002818,0.021826,0.731847,0.246739,0.278842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31921,Z,1.0,0.990123,0.947073,0.975169,0.988861,0.955055,0.558939,0.771277,0.610406,0.898910,0.979329,0.978743,0.962110,0.945741,0.563043,0.970838,0.451296
31922,Z,1.0,0.994833,0.948692,0.985982,0.983580,0.952012,0.557417,0.805806,0.603994,0.896443,0.979469,0.979442,0.960547,0.947737,0.559021,0.980647,0.456260
31923,Z,1.0,0.991369,0.946088,0.979606,0.990277,0.955284,0.554884,0.813210,0.609567,0.916302,0.978193,0.978548,0.959780,0.945149,0.572716,0.931675,0.450010
31924,Z,1.0,0.988719,0.947758,0.980312,0.984036,0.951268,0.559023,0.792340,0.612868,0.904120,0.980060,0.982656,0.962888,0.946587,0.556630,0.989731,0.459920


In [102]:
def standardize_data(data):
    for col in data.columns:
        if data[col].dtype in ['int64', 'float64']:
            data[col] = (data[col] - data[col].mean()) / data[col].std()
    return data
standardize_data(data)

,label,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
0,A,0.746507,0.260053,-2.881087,-1.494090,-1.214829,-1.260307,0.261963,-2.266637,0.395576,-0.020441,-2.734819,-1.520294,-1.375086,-1.773751,1.193835,-0.943952,-0.502484
1,A,0.746507,0.501576,-2.486072,-1.285229,-1.136839,-1.315051,0.516640,-1.705808,0.442623,0.149310,-2.803824,-1.641765,-1.404207,-1.863064,1.481318,-0.473487,-0.597342
2,A,0.746507,0.499605,-2.487885,-1.293898,-1.154553,-1.339117,0.554621,-1.852942,0.506514,0.652154,-2.817381,-1.654378,-1.432571,-1.919528,1.345697,0.242758,-0.474023
3,A,0.746507,0.275554,-2.550820,-1.327545,-1.200460,-1.366169,0.691054,-2.093398,0.645777,-0.019206,-2.838687,-1.824638,-1.680078,-2.057740,2.599985,-1.233523,-0.866285
4,A,0.746507,0.029938,-2.676411,-1.472487,-1.359990,-1.388292,1.505131,-0.958262,1.021355,0.470606,-3.019237,-1.835010,-1.743475,-2.032825,0.996318,-2.067004,-0.614972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31921,Z,0.746507,0.936910,0.598485,1.340646,1.637107,1.457368,-0.782281,1.044435,0.233285,0.283318,0.762210,1.552161,2.172011,1.840468,-0.769709,2.089385,0.259232
31922,Z,0.746507,0.983307,0.604782,1.373434,1.621077,1.448065,-0.791930,1.202974,0.211113,0.244537,0.762798,1.554689,2.165630,1.848834,-0.811787,2.145686,0.284396
31923,Z,0.746507,0.949182,0.594653,1.354101,1.641405,1.458067,-0.807987,1.236973,0.230383,0.556813,0.757441,1.551455,2.162499,1.837986,-0.668510,1.864584,0.252714
31924,Z,0.746507,0.923082,0.601150,1.356243,1.622461,1.445792,-0.781749,1.141145,0.241799,0.365252,0.765279,1.566318,2.175187,1.844012,-0.836801,2.197832,0.302951


In [103]:
def feature_scaling(data):
    for col in data.columns:
        if data[col].dtype in ['int64', 'float64']:
            data[col] = (data[col] - data[col].min()) / (data[col].max() - data[col].min())
    return data
feature_scaling(data)

,label,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
0,A,1.0,0.921411,0.052419,0.040353,0.049320,0.066008,0.723693,0.050150,0.657339,0.879593,0.146237,0.129636,0.093073,0.083624,0.750727,0.442390,0.301032
1,A,1.0,0.945929,0.153984,0.109230,0.075013,0.048100,0.763874,0.172294,0.670945,0.890388,0.129798,0.096066,0.085938,0.062319,0.778205,0.524351,0.282320
2,A,1.0,0.945729,0.153517,0.106371,0.069177,0.040227,0.769867,0.140249,0.689421,0.922365,0.126568,0.092580,0.078989,0.048851,0.765242,0.649131,0.306647
3,A,1.0,0.922984,0.137336,0.095275,0.054053,0.031377,0.791392,0.087880,0.729695,0.879671,0.121492,0.045527,0.018350,0.015882,0.885132,0.391943,0.229265
4,A,1.0,0.898051,0.105044,0.047477,0.001498,0.024140,0.919832,0.335104,0.838308,0.910820,0.078480,0.042661,0.002818,0.021826,0.731847,0.246739,0.278842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31921,Z,1.0,0.990123,0.947073,0.975169,0.988861,0.955055,0.558939,0.771277,0.610406,0.898910,0.979329,0.978743,0.962110,0.945741,0.563043,0.970838,0.451296
31922,Z,1.0,0.994833,0.948692,0.985982,0.983580,0.952012,0.557417,0.805806,0.603994,0.896443,0.979469,0.979442,0.960547,0.947737,0.559021,0.980647,0.456260
31923,Z,1.0,0.991369,0.946088,0.979606,0.990277,0.955284,0.554884,0.813210,0.609567,0.916302,0.978193,0.978548,0.959780,0.945149,0.572716,0.931675,0.450010
31924,Z,1.0,0.988719,0.947758,0.980312,0.984036,0.951268,0.559023,0.792340,0.612868,0.904120,0.980060,0.982656,0.962888,0.946587,0.556630,0.989731,0.459920


In [104]:

# inspect the column names to identify wrist coordinate labels
print("Column sample:", data.columns.tolist()[:20])


Column sample: ['label', 'both_hands', 'thumb_Left', 'index_finger_Left', 'middle_finger_Left', 'ring_finger_Left', 'pinky_Left', 'palm_angle_Left_left', 'palm_angle_Left_right', 'hand_Left_ground_angle', 'thumb_Right', 'index_finger_Right', 'middle_finger_Right', 'ring_finger_Right', 'pinky_Right', 'palm_angle_Right_left', 'palm_angle_Right_right', 'hand_Right_ground_angle']


In [106]:
# function to convert absolute landmark coordinates into relative ones based on wrist
# here prefixes are guessed from column names above ('left_hand').

def convert_to_relative(df, hand_prefix='left_hand'):
    # identify wrist coordinate columns
    wrist_x_col = f'{hand_prefix}_wrist_x'
    wrist_y_col = f'{hand_prefix}_wrist_y'
    wrist_z_col = f'{hand_prefix}_wrist_z'
    
    # loop through all columns and convert to relative coordinates
    for col in df.columns:
        if col.startswith(hand_prefix) and col not in [wrist_x_col, wrist_y_col, wrist_z_col]:
            if col.endswith('_x'):
                df[col] = df[col] - df[wrist_x_col]
            elif col.endswith('_y'):
                df[col] = df[col] - df[wrist_y_col]
            elif col.endswith('_z'):
                df[col] = df[col] - df[wrist_z_col]
    
    return df
data = convert_to_relative(data, hand_prefix='left_hand')

In [112]:
def normalize_outliers(data, column):
    for col in data.columns:
        if data[col].dtype in ['int64', 'float64']:
            Q1 = data[col].quantile(0.25)
            Q3 = data[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            data[col] = np.where(data[col] < lower_bound, lower_bound, data[col])
            data[col] = np.where(data[col] > upper_bound, upper_bound, data[col])
    return data
normalize_outliers(data, 'x1')

,label,both_hands,thumb_Left,index_finger_Left,middle_finger_Left,ring_finger_Left,pinky_Left,palm_angle_Left_left,palm_angle_Left_right,hand_Left_ground_angle,thumb_Right,index_finger_Right,middle_finger_Right,ring_finger_Right,pinky_Right,palm_angle_Right_left,palm_angle_Right_right,hand_Right_ground_angle
0,A,1.0,0.921411,0.156275,0.040353,0.049320,0.066008,0.723693,0.050150,0.657339,0.879593,0.552258,0.129636,0.093073,0.083624,0.750727,0.442390,0.301032
1,A,1.0,0.945929,0.156275,0.109230,0.075013,0.048100,0.763874,0.172294,0.670945,0.890388,0.552258,0.096066,0.085938,0.062319,0.778205,0.524351,0.282320
2,A,1.0,0.945729,0.156275,0.106371,0.069177,0.040227,0.769867,0.140249,0.689421,0.922365,0.552258,0.092580,0.078989,0.048851,0.765242,0.649131,0.306647
3,A,1.0,0.922984,0.156275,0.095275,0.054053,0.031377,0.791392,0.087880,0.729695,0.879671,0.552258,0.045527,0.078070,0.015882,0.778676,0.391943,0.229265
4,A,1.0,0.898051,0.156275,0.047477,0.001498,0.024140,0.919832,0.335104,0.838308,0.910820,0.552258,0.042661,0.078070,0.021826,0.731847,0.246739,0.278842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31921,Z,1.0,0.990123,0.947073,0.975169,0.988861,0.955055,0.558939,0.771277,0.610406,0.898910,0.979329,0.957969,0.641107,0.945741,0.563043,0.970838,0.451296
31922,Z,1.0,0.994833,0.948692,0.985982,0.983580,0.952012,0.557417,0.805806,0.603994,0.896443,0.979469,0.957969,0.641107,0.947737,0.559021,0.980647,0.456260
31923,Z,1.0,0.991369,0.946088,0.979606,0.990277,0.955284,0.554884,0.813210,0.609567,0.916302,0.978193,0.957969,0.641107,0.945149,0.572716,0.931675,0.450010
31924,Z,1.0,0.988719,0.947758,0.980312,0.984036,0.951268,0.559023,0.792340,0.612868,0.904120,0.980060,0.957969,0.641107,0.946587,0.556630,0.989731,0.459920


In [114]:
def train_test_split(data, target_column, test_size=0.2, random_state=42):
    X = data.drop(columns=[target_column])
    y = data[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    return X_train, X_test, y_train, y_test

train_test_split(data, target_column='label', test_size=0.2, random_state=42)  

KeyError: "[('A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A